In [ ]:
"""
crawling.py — 온라인 서점 베스트셀러 크롤링 및 상세정보 수집

사용법:
    python crawling.py

입력: 없음 (온라인 서점 API 직접 호출)
출력: BookStore_{genre}_{n}_enriched({장르}).json (장르별 베스트셀러 상세 데이터)

주요 처리 내용:
    - 장르별(경제/경영, 시/에세이, 자기계발, 소설, 인문) 베스트셀러 목록 수집
    - 각 책의 상세페이지에서 서지정보, 목차, 저자소개, 리뷰, AI 리뷰 요약 등 수집
    - 리뷰는 페이지네이션으로 최대 1000개까지 전체 수집
    - 중간 저장(save_every)으로 대량 수집 중단 시 이어서 진행 가능

참고 사항:
    - 코드 및 데이터 필드 내 실제 서점명은 저작권 및 보안을 위해
      익명화했습니다 (영문: BookStore / 국문: 북스토어)
    - 이 코드는 특정 시점의 대상 사이트 구조(API 엔드포인트, HTML 셀렉터)에
      의존하므로, 사이트 개편 이후에는 그대로 실행해도 동일한 결과가
      재현되지 않을 수 있습니다. 수집 방법론을 보여주기 위한 참고용입니다.
"""

import requests
import json
import re
import time
import os
from bs4 import BeautifulSoup


# =========================================================
# 0. 세션 & 헤더 설정
# =========================================================
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "Accept-Encoding": "gzip, deflate, br",
    "Referer": "https://www.BookStorebook.co.kr/",
    "Connection": "keep-alive",
}
session = requests.Session()


# =========================================================
# 1. 베스트셀러 목록 API
# =========================================================
def fetch_bestseller_page(session, headers, clst_code, page, per=20):
    api_url = "https://BookStore.co.kr/api/gw/best/v2/best-seller/online"
    params = {
        "page": page,
        "per": per,
        "saleCmdtClstCode": clst_code,
        "soldOutExcludeYn": "N",
        "saleCmdtDsplDvsnCode": "KOR",
        "period": "002",
        "dsplDvsnCode": "001",
        "dsplTrgtDvsnCode": "004",
    }
    res = session.get(api_url, headers=headers, params=params, timeout=10)
    res.raise_for_status()
    return res.json()


def extract_book_summary(item):
    info = item["product"]["productInfo"]
    review = item["product"]["reviewInfo"]
    return {
        "rank": item["prstRnkn"],
        "saleCmdtid": info["saleCmdtid"],
        "isbn": info["cmdtcode"],
        "title": info["cmdtName"],
        "publisher": info["pbcmName"],
        "author": info["chrcName"],
        "release_date": info["rlseDate"],
        "genre_code": info["saleCmdtClstCode"],
        "genre_name": info["saleCmdtClstName"],
        "description": info["anntCntt"],
        "rating_value": review["score"],
        "rating_count": review["count"],
    }


def collect_genre_bestsellers(session, headers, clst_code, genre_label, top_n=100, per=20):
    books = []
    pages_needed = (top_n + per - 1) // per

    for page in range(1, pages_needed + 1):
        data = fetch_bestseller_page(session, headers, clst_code, page, per)
        items = data["data"]["bestSeller"]
        if not items:
            break
        for item in items:
            books.append(extract_book_summary(item))
        print(f"  [{genre_label}] page {page} 완료 ({len(items)}권)")
        time.sleep(0.5)

    return books[:top_n]


# =========================================================
# 2. 상세페이지 파싱 헬퍼 함수들
# =========================================================
def parse_spec_table(table):
    spec = {}
    for row in table.find_all("tr"):
        th = row.find("th")
        td = row.find("td")
        if not (th and td):
            continue
        key = th.get_text(strip=True)
        td_copy = BeautifulSoup(str(td), "lxml")
        for btn in td_copy.find_all("button"):
            btn.decompose()
        value = td_copy.get_text(separator=" ", strip=True)
        value = re.sub(r"\s+", " ", value).strip()
        spec[key] = value
    return spec


def parse_author_info(soup):
    authors = []
    person_section = soup.find("div", class_="product_person")
    if not person_section:
        return authors

    for box in person_section.find_all("div", class_="round_gray_box"):
        title_h3 = box.find("h3", class_="title_heading")
        role = None
        name = None
        if title_h3:
            prefix = title_h3.find("span", class_="title_prefix")
            link = title_h3.find("a", class_="person_link")
            role = prefix.get_text(strip=True) if prefix else None
            name = link.get("data-author-name") if link else None

        bio_p = box.find("p", class_="info_text")
        bio = bio_p.get_text(strip=True) if bio_p else None

        authors.append({"role": role, "name": name, "bio": bio})
    return authors


def parse_detail_images(soup):
    images = []
    for img in soup.find_all("img"):
        src = img.get("src") or img.get("data-src")
        if src and "illustrate" in src:
            images.append(src)
    return images


def parse_simple_section(soup, wrapper_class):
    section = soup.find("div", class_=wrapper_class)
    if not section:
        return None
    info_text = section.find("p", class_="info_text")
    if not info_text:
        return None
    return info_text.get_text(separator="\n", strip=True)


def parse_category_paths(soup):
    paths = []
    category_list = soup.find("ul", class_="intro_category_list")
    if not category_list:
        return paths
    for li in category_list.find_all("li", class_="category_list_item"):
        links = li.find_all("a", class_="intro_category_link")
        path = [a.get_text(strip=True) for a in links]
        if path:
            paths.append(" > ".join(path))
    return paths


def parse_awards(soup):
    awards = []
    award_section = soup.find("div", class_="intro_award")
    if not award_section:
        return awards
    for li in award_section.find_all("li", class_="text_award_item"):
        text = li.get_text(separator=" ", strip=True)
        text = re.sub(r"\s+", " ", text).strip()
        awards.append(text)
    return awards


def parse_recommendations(soup):
    recommends = []
    recommend_section = soup.find("div", class_="book_recommend")
    if not recommend_section:
        return recommends
    for li in recommend_section.find_all("li", class_="recommend_item"):
        name_tag = li.find("a", class_="title_heading")
        quote_tag = li.find("p", class_="info_text")
        name = name_tag.get_text(strip=True) if name_tag else None
        quote = quote_tag.get_text(strip=True) if quote_tag else None
        if name or quote:
            recommends.append({"name": name, "quote": quote})
    return recommends


# =========================================================
# 3. 리뷰 API + AI 리뷰 요약 API
# =========================================================
def fetch_all_reviews(saleCmdtid, session, headers, max_reviews=1000):
    """전체 리뷰 가져오기. pageLimit을 크게 잡아서 호출 횟수 최소화.
    total_review_count로 실제 전체 리뷰 수를 같이 보존(브론즈 원칙)."""
    review_url = "https://product.BookStorebook.co.kr/api/review/list"
    all_reviews = []
    total_available = 0
    page = 1
    page_limit = 1000

    while len(all_reviews) < max_reviews:
        params = {
            "page": page,
            "pageLimit": page_limit,
            "reviewSort": "002",
            "revwPatrCode": "000",
            "saleCmdtids": saleCmdtid,
            "webToonYsno": "N",
            "allYsno": "N",
            "revwSummeryYn": "Y",
            "saleCmdtid": saleCmdtid,
        }
        res = session.get(review_url, headers=headers, params=params, timeout=10)
        res.raise_for_status()
        data = res.json()

        reviews = data["data"]["reviewList"]
        total_available = data["data"]["totalCount"]
        if not reviews:
            break

        for r in reviews:
            all_reviews.append({
                "rating": r["revwRvgr"],
                "content": r["revwCntt"],
                "created_at": r["cretDttm"],
                "emotion_tag": r.get("revwEmtnKywrName"),
            })

        if len(all_reviews) >= total_available:
            break
        page += 1

    return {
        "reviews": all_reviews[:max_reviews],
        "total_review_count": total_available,
    }


def fetch_ai_review_summary(saleCmdtid, session, headers):
    """AI 리뷰 요약 전용 API 호출"""
    summary_url = "https://BookStore.co.kr/api/gw/pdt/review/summary"
    params = {"saleCmdtid": saleCmdtid}

    try:
        res = session.get(summary_url, headers=headers, params=params, timeout=10)
        res.raise_for_status()
        data = res.json().get("data")

        if not data or not data.get("summ_revw_cntt"):
            return None

        keywords_raw = data.get("summ_revw_kywr_cntt", "")
        tags = [f"#{k.strip()}" for k in keywords_raw.split(",") if k.strip()]

        return {
            "lead": data.get("summ_oli_cntt"),
            "detail": data.get("summ_revw_cntt"),
            "tags": tags,
        }
    except Exception:
        return None


# =========================================================
# 4. 상세페이지 종합 함수
# =========================================================
def get_book_detail(product_url, list_description=None, session=None):
    sess = session or requests.Session()
    res = sess.get(product_url, headers=headers, timeout=10)
    soup = BeautifulSoup(res.text, "lxml")

    sale_cmdtid = product_url.rstrip("/").split("/")[-1]

    result = {"url": product_url}

    # 4-1. ld+json
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string)
        except (json.JSONDecodeError, TypeError):
            continue
        if data.get("@type") == "Book":
            result["title"] = data.get("name")
            result["image"] = data.get("image")
            result["genre"] = data.get("genre")
            result["keywords"] = data.get("keywords")
            result["author"] = data.get("author", {}).get("name")
            result["publisher"] = data.get("publisher", {}).get("name")
            rating = data.get("aggregateRating", {})
            result["rating_value"] = rating.get("ratingValue")
            result["rating_count"] = rating.get("ratingCount")
            work = data.get("workExample", [{}])[0]
            result["isbn"] = work.get("isbn")
            result["date_published"] = work.get("datePublished")
            offer = work.get("potentialAction", {}).get("expectsAcceptanceOf", {})
            result["price"] = offer.get("Price")

    # 4-2. 책소개 전문
    intro_div = soup.find("div", class_="intro_bottom")
    if intro_div:
        texts = intro_div.find_all("div", class_="info_text")
        full_text = "\n".join(t.get_text(separator="\n", strip=True) for t in texts)
        result["description_full"] = full_text if full_text else list_description
    else:
        result["description_full"] = list_description

    # 4-3. 목차
    toc_section = soup.find("div", class_="book_contents")
    if toc_section:
        toc_items = toc_section.find_all("li", class_="book_contents_item")
        result["table_of_contents"] = "\n".join(
            item.get_text(separator="\n", strip=True) for item in toc_items
        )
    else:
        result["table_of_contents"] = None

    # 4-4. 스펙 테이블
    spec_table = None
    for table in soup.find_all("table"):
        caption = table.find("caption")
        if caption and "상품정보" in caption.get_text():
            spec_table = table
            break
    result["spec"] = parse_spec_table(spec_table) if spec_table else {}

    # 4-5. 작가 정보
    result["authors"] = parse_author_info(soup)

    # 4-6. 작가의 말
    result["writer_words"] = parse_simple_section(soup, "writer_words")

    # 4-7. 책 속으로
    result["book_excerpt"] = parse_simple_section(soup, "book_inside")

    # 4-7-1. 카테고리 경로
    result["category_paths"] = parse_category_paths(soup)

    # 4-7-2. 수상내역/미디어추천
    result["awards"] = parse_awards(soup)

    # 4-7-3. 추천사
    result["recommendations"] = parse_recommendations(soup)

    # 4-7-4. AI 리뷰 요약
    result["ai_review_summary"] = fetch_ai_review_summary(sale_cmdtid, sess, headers)

    # 4-8. 상세 이미지 URL
    result["detail_image_urls"] = parse_detail_images(soup)

    # 4-9. 리뷰 (전체, total_review_count 같이 보존)
    try:
        review_data = fetch_all_reviews(sale_cmdtid, sess, headers, max_reviews=1000)
        result["reviews"] = review_data["reviews"]
        result["total_review_count"] = review_data["total_review_count"]
    except Exception as e:
        result["reviews"] = []
        result["total_review_count"] = None
        print(f"  [리뷰 수집 실패] {sale_cmdtid}: {e}")

    return result


# =========================================================
# 5. 대량 보강 함수
# =========================================================
def enrich_books_with_details(book_list, session, save_path="enriched_books.json", save_every=50):
    if os.path.exists(save_path):
        with open(save_path, "r", encoding="utf-8") as f:
            enriched = json.load(f)
        done_ids = {b["saleCmdtid"] for b in enriched}
        print(f"기존 진행 파일 발견: {len(enriched)}권 이미 완료됨, 이어서 진행합니다.")
    else:
        enriched = []
        done_ids = set()

    failed = []

    for i, book in enumerate(book_list):
        cmdtid = book["saleCmdtid"]
        if cmdtid in done_ids:
            continue

        url = f"https://product.BookStore.co.kr/detail/{cmdtid}"
        list_desc = book.get("description")

        try:
            detail = get_book_detail(url, list_description=list_desc, session=session)
            merged = {**book, **detail}
            merged.pop("description", None)
            enriched.append(merged)
        except Exception as e:
            print(f"  [실패] {cmdtid} ({book.get('title')}): {e}")
            failed.append({"saleCmdtid": cmdtid, "title": book.get("title"), "error": str(e)})

        if (i + 1) % 20 == 0:
            print(f"진행: {i + 1}/{len(book_list)} (완료 {len(enriched)}, 실패 {len(failed)})")

        if (i + 1) % save_every == 0:
            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(enriched, f, ensure_ascii=False, indent=2)
            print(f"  → 중간 저장 완료 ({len(enriched)}권)")

        time.sleep(0.5)

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(enriched, f, ensure_ascii=False, indent=2)

    if failed:
        with open("enrich_failed.json", "w", encoding="utf-8") as f:
            json.dump(failed, f, ensure_ascii=False, indent=2)
        print(f"\n실패한 {len(failed)}권은 enrich_failed.json에 기록했습니다.")

    print(f"\n최종 완료: {len(enriched)}권")
    return enriched

# =========================================================
# 장르별 베스트셀러 수집 (경제/경영, 시/에세이, 자기계발, 소설, 인문)
# =========================================================
GENRES = [
    {"code": "13", "label": "경제/경영", "filename": "BookStore_business_1000_enriched(경제경영).json"},
    {"code": "03", "label": "시/에세이", "filename": "BookStore_essay_1000_enriched(시에세이).json"},
    {"code": "15", "label": "자기계발", "filename": "BookStore_selfhelp_1000_enriched(자기계발).json"},
    {"code": "01", "label": "소설",     "filename": "BookStore_novel_1000_enriched(소설).json"},
    {"code": "05", "label": "인문",     "filename": "BookStore_humanities_1000_enriched(인문).json"},
]

enriched_by_genre = {}

for genre in GENRES:
    code, label, filename = genre["code"], genre["label"], genre["filename"]

    print(f"=== {label}: 1단계 목록 수집 ===")
    books = collect_genre_bestsellers(session, headers, code, label, top_n=100)
    print(f"목록 수집 완료: {len(books)}권\n")

    print(f"=== {label}: 2단계 상세정보 보강 ===")
    enriched = enrich_books_with_details(
        books,
        session,
        save_path=filename,
    )

    enriched_by_genre[label] = enriched